In [2]:
import pandas as pd

In [3]:
train = pd.read_csv("/content/drive/MyDrive/KDU/Thesis/Dataset/tram_train.tsv", sep='\t')
test = pd.read_csv("/content/drive/MyDrive/KDU/Thesis/Dataset/tram_test.tsv", sep='\t')
dev = pd.read_csv("/content/drive/MyDrive/KDU/Thesis/Dataset/tram_dev.tsv", sep='\t')

In [4]:
from sklearn.preprocessing import MultiLabelBinarizer
import ast

for df in (train, dev, test):
    df["y"] = df["labels"].map(lambda x: sorted(set(ast.literal_eval(x))))  # dedup here

mlb = MultiLabelBinarizer()
mlb.fit(train["y"])          # fit only on train classes
Y_train = mlb.transform(train["y"])
Y_dev   = mlb.transform(dev["y"])    # labels unseen in train are silently dropped
Y_test  = mlb.transform(test["y"])

/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['T1055.011', 'T1216', 'T1547.005', 'T1569.001'] will be ignored
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['T1011', 'T1134.005', 'T1218.002', 'T1485', 'T1547.005', 'T1548.001', 'T1548.003', 'T1552.004', 'T1556.002', 'T1574.010'] will be ignored
  warnings.warn(


In [5]:
# print(Y_train[:5])
# print(Y_test[:5])
# print(Y_dev[:5])

In [6]:
import os
import random
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import f1_score, precision_score, recall_score
from tqdm.auto import tqdm

# --- HELPER FUNCTIONS (implemented in-place to avoid missing imports) ---

def dedupe_texts(df):
    # Keep first occurrence of duplicates based on text
    return df.drop_duplicates(subset=["text"]).reset_index(drop=True)

def compute_pos_weight(Y_train):
    # pos_weight = (negative_instances) / (positive_instances)
    # Clip positive instances count at 1 to avoid division by zero
    pos_counts = Y_train.sum(axis=0)
    neg_counts = Y_train.shape[0] - pos_counts
    pos_counts = np.clip(pos_counts, 1, None)
    return neg_counts / pos_counts

def best_global_threshold(y_true, y_probs):
    # Find a single optimal decision threshold on dev set
    best_f1 = -1.0
    best_thresh = 0.5
    for thresh in np.arange(0.1, 0.9, 0.05):
        y_pred = (y_probs >= thresh).astype(int)
        score = f1_score(y_true, y_pred, average="micro", zero_division=0)
        if score > best_f1:
            best_f1 = score
            best_thresh = thresh
    return best_thresh

def best_per_class_thresholds(y_true, y_probs):
    # Find class-specific optimal decision thresholds
    num_classes = y_true.shape[1]
    thresholds = []
    for i in range(num_classes):
        best_f1 = -1.0
        best_thresh = 0.5
        for thresh in np.arange(0.1, 0.9, 0.05):
            y_pred = (y_probs[:, i] >= thresh).astype(int)
            score = f1_score(y_true[:, i], y_pred, average="binary", zero_division=0)
            if score > best_f1:
                best_f1 = score
                best_thresh = thresh
        thresholds.append(best_thresh)
    return thresholds

def apply_thresholds(probs, thresholds):
    if isinstance(thresholds, list):
        # Per-class thresholds
        preds = np.zeros_like(probs)
        for i, t in enumerate(thresholds):
            preds[:, i] = (probs[:, i] >= t).astype(int)
        return preds
    else:
        # Global threshold
        return (probs >= thresholds).astype(int)

def evaluate(y_true, y_pred):
    micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    precision = precision_score(y_true, y_pred, average="micro", zero_division=0)
    recall = recall_score(y_true, y_pred, average="micro", zero_division=0)
    return {
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "micro_precision": precision,
        "micro_recall": recall
    }

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

# --- PYTORCH DATASET ---
class TramDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = list(texts)
        self.labels = labels  # numpy array, multi-hot
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], truncation=True, max_length=self.max_length,
            padding="max_length", return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

# --- EPOCH RUNNER WITH REAL-TIME PROGRESS BAR ---
def run_epoch(model, loader, loss_fn, device, optimizer=None, scheduler=None, desc="Evaluating"):
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()
    total_loss, all_probs, all_labels = 0.0, [], []

    # Use tqdm to show step-by-step training updates
    progress_bar = tqdm(loader, desc=desc, leave=False)
    for batch in progress_bar:
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch.pop("labels")
        with torch.set_grad_enabled(train_mode):
            out = model(**batch)
            loss = loss_fn(out.logits, labels)
            if train_mode:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
        total_loss += loss.item() * labels.size(0)
        all_probs.append(torch.sigmoid(out.logits).detach().cpu().numpy())
        all_labels.append(labels.detach().cpu().numpy())

        # Update progress bar with current batch loss
        progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})

    probs = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)
    return total_loss / len(loader.dataset), probs, labels

In [7]:
def train_model(
    model_name: str,
    output_dir: str,
    seed: int = 42,
    epochs: int = 5,
    batch_size: int = 16,
    lr: float = 2e-5,
    max_length: int = 128,
    warmup_ratio: float = 0.1,
    use_pos_weight: bool = True,
    per_class_threshold: bool = False
):
    set_seed(seed)
    os.makedirs(output_dir, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ---- process and clean training data ----
    # We utilize 'train', 'dev', and 'test' DataFrames already loaded in previous cells.
    clean_train = dedupe_texts(train)

    num_classes = len(mlb.classes_)
    print(f"[{model_name} seed={seed}] train={len(clean_train)} "
          f"dev={len(dev)} test={len(test)} classes={num_classes}")

    Y_train_clean = mlb.transform(clean_train["y"])
    Y_dev_clean = mlb.transform(dev["y"])
    Y_test_clean = mlb.transform(test["y"])

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    train_ds = TramDataset(clean_train["text"], Y_train_clean, tokenizer, max_length)
    dev_ds = TramDataset(dev["text"], Y_dev_clean, tokenizer, max_length)
    test_ds = TramDataset(test["text"], Y_test_clean, tokenizer, max_length)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    dev_loader = DataLoader(dev_ds, batch_size=batch_size * 2, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size * 2, shuffle=False)

    # ---- model ----
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=num_classes,
        problem_type="multi_label_classification",
    ).to(device)

    pos_weight = None
    if use_pos_weight:
        pos_weight = torch.tensor(compute_pos_weight(Y_train_clean),
                                   dtype=torch.float, device=device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    total_steps = len(train_loader) * epochs
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(total_steps * warmup_ratio),
        num_training_steps=total_steps,
    )

    # ---- train with dev-based best-checkpoint selection ----
    best_dev_micro_f1 = -1.0
    best_state = None
    for epoch in range(1, epochs + 1):
        train_loss, _, _ = run_epoch(model, train_loader, loss_fn, device,
                                      optimizer, scheduler)
        dev_loss, dev_probs, dev_labels = run_epoch(model, dev_loader, loss_fn, device)
        dev_thresh = best_global_threshold(dev_labels, dev_probs)
        dev_preds = apply_thresholds(dev_probs, dev_thresh)
        dev_metrics = evaluate(dev_labels, dev_preds)
        print(f"  epoch {epoch}: train_loss={train_loss:.4f} dev_loss={dev_loss:.4f} "
              f"dev_micro_f1={dev_metrics['micro_f1']:.4f} dev_macro_f1={dev_metrics['macro_f1']:.4f}")
        if dev_metrics["micro_f1"] > best_dev_micro_f1:
            best_dev_micro_f1 = dev_metrics["micro_f1"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)

    # ---- final threshold tuning on dev with the BEST checkpoint ----
    _, dev_probs, dev_labels = run_epoch(model, dev_loader, loss_fn, device)
    if per_class_threshold:
        thresholds = best_per_class_thresholds(dev_labels, dev_probs)
    else:
        thresholds = best_global_threshold(dev_labels, dev_probs)

    # ---- test set ----
    _, test_probs, test_labels = run_epoch(model, test_loader, loss_fn, device)
    test_preds = apply_thresholds(test_probs, thresholds)
    test_metrics = evaluate(test_labels, test_preds)
    print(f"  TEST: {test_metrics}")

    # ---- save everything the SHAP phase and results table will need ----
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    np.save(os.path.join(output_dir, "mlb_classes.npy"), mlb.classes_)
    np.save(os.path.join(output_dir, "test_probs.npy"), test_probs)
    np.save(os.path.join(output_dir, "test_preds.npy"), test_preds)
    np.save(os.path.join(output_dir, "test_labels.npy"), test_labels)
    with open(os.path.join(output_dir, "thresholds.json"), "w") as f:
        json.dump(np.asarray(thresholds).tolist(), f)
    with open(os.path.join(output_dir, "metrics.json"), "w") as f:
        json.dump({
            "model_name": model_name, "seed": seed,
            "dev_best_micro_f1": best_dev_micro_f1, "test": test_metrics,
        }, f, indent=2)

    print(f"Saved checkpoint + metrics to {output_dir}")

In [8]:
# Map the code to use the correct 'text1' column name from the loaded dataset
def dedupe_texts_fixed(df):
    return df.drop_duplicates(subset=["text1"]).reset_index(drop=True)

class TramDatasetFixed(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = list(texts)
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], truncation=True, max_length=self.max_length,
            padding="max_length", return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

def train_model_fixed(
    model_name: str,
    output_dir: str,
    seed: int = 42,
    epochs: int = 5,
    batch_size: int = 16,
    lr: float = 2e-5,
    max_length: int = 128,
    warmup_ratio: float = 0.1,
    use_pos_weight: bool = True,
    per_class_threshold: bool = False
):
    set_seed(seed)
    os.makedirs(output_dir, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    clean_train = dedupe_texts_fixed(train)

    num_classes = len(mlb.classes_)
    print(f"[{model_name} seed={seed}] train={len(clean_train)} "
          f"dev={len(dev)} test={len(test)} classes={num_classes}")

    Y_train_clean = mlb.transform(clean_train["y"])
    Y_dev_clean = mlb.transform(dev["y"])
    Y_test_clean = mlb.transform(test["y"])

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    train_ds = TramDatasetFixed(clean_train["text1"], Y_train_clean, tokenizer, max_length)
    dev_ds = TramDatasetFixed(dev["text1"], Y_dev_clean, tokenizer, max_length)
    test_ds = TramDatasetFixed(test["text1"], Y_test_clean, tokenizer, max_length)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    dev_loader = DataLoader(dev_ds, batch_size=batch_size * 2, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size * 2, shuffle=False)

    # ---- model ----
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=num_classes,
        problem_type="multi_label_classification",
    ).to(device)

    pos_weight = None
    if use_pos_weight:
        pos_weight = torch.tensor(compute_pos_weight(Y_train_clean),
                                   dtype=torch.float, device=device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    total_steps = len(train_loader) * epochs
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(total_steps * warmup_ratio),
        num_training_steps=total_steps,
    )

    best_dev_micro_f1 = -1.0
    best_state = None
    for epoch in range(1, epochs + 1):
        train_loss, _, _ = run_epoch(
            model, train_loader, loss_fn, device, optimizer, scheduler, desc=f"Epoch {epoch}/{epochs} [Train]"
        )
        dev_loss, dev_probs, dev_labels = run_epoch(
            model, dev_loader, loss_fn, device, desc=f"Epoch {epoch}/{epochs} [Dev]"
        )
        dev_thresh = best_global_threshold(dev_labels, dev_probs)
        dev_preds = apply_thresholds(dev_probs, dev_thresh)
        dev_metrics = evaluate(dev_labels, dev_preds)
        print(f"Epoch {epoch} finished: train_loss={train_loss:.4f} dev_loss={dev_loss:.4f} "
              f"dev_micro_f1={dev_metrics['micro_f1']:.4f} dev_macro_f1={dev_metrics['macro_f1']:.4f}")
        if dev_metrics["micro_f1"] > best_dev_micro_f1:
            best_dev_micro_f1 = dev_metrics["micro_f1"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)

    _, dev_probs, dev_labels = run_epoch(model, dev_loader, loss_fn, device, desc="Final Threshold Tuning [Dev]")
    if per_class_threshold:
        thresholds = best_per_class_thresholds(dev_labels, dev_probs)
    else:
        thresholds = best_global_threshold(dev_labels, dev_probs)

    _, test_probs, test_labels = run_epoch(model, test_loader, loss_fn, device, desc="Evaluating [Test]")
    test_preds = apply_thresholds(test_probs, thresholds)
    test_metrics = evaluate(test_labels, test_preds)
    print(f"  TEST RESULTS: {test_metrics}")

    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    np.save(os.path.join(output_dir, "mlb_classes.npy"), mlb.classes_)
    np.save(os.path.join(output_dir, "test_probs.npy"), test_probs)
    np.save(os.path.join(output_dir, "test_preds.npy"), test_preds)
    np.save(os.path.join(output_dir, "test_labels.npy"), test_labels)
    with open(os.path.join(output_dir, "thresholds.json"), "w") as f:
        json.dump(np.asarray(thresholds).tolist(), f)
    with open(os.path.join(output_dir, "metrics.json"), "w") as f:
        json.dump({
            "model_name": model_name, "seed": seed,
            "dev_best_micro_f1": best_dev_micro_f1, "test": test_metrics,
        }, f, indent=2)

    print(f"Saved checkpoint + metrics to {output_dir}")

# Run the corrected training wrapper
train_model_fixed(model_name="roberta-base", output_dir="runs/roberta", seed=42)

Using device: cuda
[roberta-base seed=42] train=3468 dev=604 test=725 classes=180


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (20 outputs left out)

Evaluating [Test]:   0%|          | 0/23 [00:00<?, ?it/s]

  TEST RESULTS: {'micro_f1': 0.2983425414364641, 'macro_f1': 0.1377050549020432, 'micro_precision': 0.3083756345177665, 'micro_recall': 0.28894173602853745}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved checkpoint + metrics to runs/roberta
